# Policyscope: сравнение оценщиков с oracle (RU)

Цель: увидеть, как Replay / IPS / SNIPS / DM / DR / SNDR / Switch-DR ведут себя относительно oracle в контролируемых synthetic-сценариях.

In [1]:
import pathlib
import sys

cwd = pathlib.Path.cwd()
for cand in [cwd / 'src', cwd.parent / 'src', pathlib.Path('/workspace/OffPolicyLab/src')]:
    if cand.exists() and str(cand) not in sys.path:
        sys.path.insert(0, str(cand))

import io
import logging
from contextlib import redirect_stdout, redirect_stderr

import numpy as np
import pandas as pd

from policyscope.synthetic import SynthConfig, SyntheticRecommenderEnv
from policyscope.policies import make_policy
from policyscope.data import BanditSchema, LoggedBanditDataset
from policyscope.comparison import compare_policies

logging.getLogger().setLevel(logging.WARNING)
pd.set_option('display.max_columns', 80)

def quiet_call(fn, *args, **kwargs):
    out, err = io.StringIO(), io.StringIO()
    with redirect_stdout(out), redirect_stderr(err):
        return fn(*args, **kwargs)

## 1) Вспомогательная функция для одного synthetic-сценария

In [2]:
def run_scenario(*, n_users, seed, epsilon_A, tau_B, include_logged_propensity):
    env = SyntheticRecommenderEnv(SynthConfig(n_users=n_users, seed=seed))
    X = env.sample_users()
    policyA = make_policy('epsilon_greedy', seed=seed, epsilon=epsilon_A)
    policyB = make_policy('softmax', seed=seed + 99, tau=tau_B)

    logs = env.simulate_logs_A(policyA, X)
    probs_A = policyA.action_probs(logs)
    logs['propensity_A'] = probs_A[np.arange(len(logs)), logs['a_A'].to_numpy()]
    if not include_logged_propensity:
        logs = logs.drop(columns=['propensity_A'])

    schema = BanditSchema(
        action_col='a_A',
        reward_col='accept',
        feature_cols=['loyal', 'age', 'risk', 'income'],
        cluster_col='user_id',
        propensity_col='propensity_A' if include_logged_propensity else None,
    )
    logged = LoggedBanditDataset(df=logs, schema=schema)

    oracle_A = env.oracle_value(policyA, X, metric='accept', n_mc=5)
    oracle_B = env.oracle_value(policyB, X, metric='accept', n_mc=5)
    oracle_delta = oracle_B - oracle_A

    rows = []
    for est in ['replay', 'ips', 'snips', 'dm', 'dr', 'sndr', 'switch_dr']:
        s = quiet_call(
            compare_policies,
            logged,
            policyB,
            estimator=est,
            target='accept',
            n_boot=3,
            alpha=0.1,
            weight_clip=20.0,
            tau=20.0,
            propensity_source='auto',
        )
        d = s.to_dict()
        diag = d.get('diagnostics', {})
        rows.append({
            'estimator': est,
            'V_B_est': d['V_B'],
            'V_B_oracle': oracle_B,
            'Delta_est': d['Delta'],
            'Delta_oracle': oracle_delta,
            'abs_error_V_B': abs(d['V_B'] - oracle_B),
            'abs_error_Delta': abs(d['Delta'] - oracle_delta),
            'replay_overlap': diag.get('replay_overlap'),
            'weight_ess_ratio': diag.get('weight_ess_ratio'),
            'trust_level': d.get('trust_level'),
        })

    table = pd.DataFrame(rows).sort_values(['abs_error_Delta', 'abs_error_V_B'])
    meta = {
        'oracle_V_A': oracle_A,
        'oracle_V_B': oracle_B,
        'oracle_Delta': oracle_delta,
        'include_logged_propensity': include_logged_propensity,
        'epsilon_A': epsilon_A,
        'tau_B': tau_B,
    }
    return meta, table

## 2) Сценарий A: относительно здоровый overlap

Logging policy A достаточно стохастическая (`epsilon_A=0.30`), поэтому взвешенные методы обычно стабильнее.

In [3]:
meta_good, table_good = run_scenario(n_users=600, seed=42, epsilon_A=0.30, tau_B=1.2, include_logged_propensity=True)
pd.Series(meta_good)

oracle_V_A                   0.532667
oracle_V_B                   0.503333
oracle_Delta                -0.029333
include_logged_propensity        True
epsilon_A                         0.3
tau_B                             1.2
dtype: object

In [4]:
table_good

,estimator,V_B_est,V_B_oracle,Delta_est,Delta_oracle,abs_error_V_B,abs_error_Delta,replay_overlap,weight_ess_ratio,trust_level
3,dm,0.490039,0.503333,-0.033294,-0.029333,0.013294,0.003961,0.311667,NaN,caution
4,dr,0.503504,0.503333,-0.019829,-0.029333,0.000171,0.009504,0.311667,0.218594,caution
6,switch_dr,0.503504,0.503333,-0.019829,-0.029333,0.000171,0.009504,0.311667,0.218594,caution
5,sndr,0.504266,0.503333,-0.019067,-0.029333,0.000933,0.010266,0.311667,0.218594,caution
1,ips,0.482158,0.503333,-0.041176,-0.029333,0.021176,0.011842,0.311667,0.218594,caution
2,snips,0.509445,0.503333,-0.013889,-0.029333,0.006111,0.015445,0.311667,0.218594,caution
0,replay,0.550802,0.503333,0.027469,-0.029333,0.047469,0.056802,0.311667,NaN,caution


## 3) Сценарий B: плохой overlap

`epsilon_A=0.02` делает policy A почти детерминированной. Это обычно ухудшает ESS и устойчивость IPS-подобных методов.

In [5]:
meta_bad, table_bad = run_scenario(n_users=600, seed=123, epsilon_A=0.02, tau_B=0.7, include_logged_propensity=True)
pd.Series(meta_bad)

oracle_V_A                      0.467
oracle_V_B                   0.533667
oracle_Delta                 0.066667
include_logged_propensity        True
epsilon_A                        0.02
tau_B                             0.7
dtype: object

In [6]:
table_bad

,estimator,V_B_est,V_B_oracle,Delta_est,Delta_oracle,abs_error_V_B,abs_error_Delta,replay_overlap,weight_ess_ratio,trust_level
0,replay,0.578431,0.533667,0.081765,0.066667,0.044765,0.015098,0.17,NaN,caution
6,switch_dr,0.539596,0.533667,0.042929,0.066667,0.005929,0.023738,0.17,0.014983,elevated_concern
3,dm,0.523768,0.533667,0.027101,0.066667,0.009899,0.039565,0.17,NaN,caution
4,dr,0.495250,0.533667,-0.001417,0.066667,0.038417,0.068084,0.17,0.014983,elevated_concern
5,sndr,0.467043,0.533667,-0.029623,0.066667,0.066623,0.096290,0.17,0.014983,elevated_concern
2,snips,0.439544,0.533667,-0.057123,0.066667,0.094123,0.123790,0.17,0.014983,elevated_concern
1,ips,0.220979,0.533667,-0.275687,0.066667,0.312687,0.342354,0.17,0.014983,elevated_concern


## 4) Сравнение logged vs estimated propensity (IPS, poor-overlap)

Показываем, как результат может меняться в неблагоприятном overlap-сценарии при смене источника propensity.

In [7]:
env = SyntheticRecommenderEnv(SynthConfig(n_users=600, seed=555))
X = env.sample_users()
policyA = make_policy('epsilon_greedy', seed=555, epsilon=0.02)
policyB = make_policy('softmax', seed=777, tau=0.7)
logs = env.simulate_logs_A(policyA, X)
probs_A = policyA.action_probs(logs)
logs['propensity_A'] = probs_A[np.arange(len(logs)), logs['a_A'].to_numpy()]

schema = BanditSchema(action_col='a_A', reward_col='accept', feature_cols=['loyal', 'age', 'risk', 'income'], cluster_col='user_id', propensity_col='propensity_A')
logged = LoggedBanditDataset(df=logs, schema=schema)

rows = []
for mode in ['logged', 'estimated']:
    s = quiet_call(compare_policies, logged, policyB, estimator='ips', target='accept', n_boot=3, alpha=0.1, propensity_source=mode)
    d = s.to_dict()
    rows.append({'propensity_mode': mode, 'V_B': d['V_B'], 'Delta': d['Delta'], 'ess_ratio': d.get('diagnostics',{}).get('weight_ess_ratio'), 'trust_level': d.get('trust_level')})

pd.DataFrame(rows)

,propensity_mode,V_B,Delta,ess_ratio,trust_level
0,logged,0.602007,0.045340,0.010262,elevated_concern
1,estimated,229.292164,228.735498,0.001773,elevated_concern


## 5) Практический вывод

- Смотрите на `Delta` вместе с `Delta_CI`/`p_value` **и** diagnostics.
- При плохом overlap: больше осторожности к IPS/SNIPS, особенно при тяжёлых хвостах весов.
- DR — разумный practical default, если nuisance-модели не провалены.
- SNDR / Switch-DR могут быть устойчивее при нестабильных весах (за счёт нормировки/переключения).
- Replay полезен как baseline на пересечении support, но может терять данные при низком overlap.